# Build the client-level table

## Load source tables

In [1]:
import pandas as pd

df1 = pd.read_csv('df_demo_clean.csv')
df2 = pd.read_csv('df_final_experiment_clients.csv')
df_events = pd.read_csv('events_kpi_table.csv')
df_visits = pd.read_csv('visits_kpi_table.csv')

## Merge demographics with experiment assignment

Inner join keeps only clients who appear in both tables (i.e., demo info exists *and* they were assigned to a group).

In [2]:
df_clients_roster = df1.merge(df2, on='client_id', how='inner')

In [3]:
# Drop clients without an assigned Variation. They don't belong to either Test or Control
df_clients_roster = df_clients_roster.dropna(subset=['Variation'])

In [4]:
# 'X' is an unclear/edge gender code. Let's fold into 'U' (unknown) for clean grouping later
df_clients_roster['gendr'] = df_clients_roster['gendr'].replace({'X': 'U'})

In [5]:
df_clients_roster

,client_id,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
0,836976,73.0,60.5,U,2.0,45105.30,6.0,9.0,Test
1,2304905,94.0,58.0,U,2.0,110860.30,6.0,9.0,Control
2,1439522,64.0,32.0,U,2.0,52467.79,6.0,9.0,Test
3,1562045,198.0,49.0,M,2.0,67454.65,3.0,6.0,Test
4,5126305,145.0,33.0,F,2.0,103671.75,0.0,3.0,Control
...,...,...,...,...,...,...,...,...,...
50482,1780858,262.0,68.5,M,3.0,372100.59,6.0,9.0,Test
50483,6967120,260.0,68.5,M,3.0,4279873.38,6.0,9.0,Control
50484,5826160,249.0,56.5,F,2.0,44837.16,2.0,5.0,Test
50485,8739285,229.0,69.5,F,2.0,44994.24,1.0,4.0,Test


In [6]:
df_events.head(50)

,client_id,visitor_id,visit_id,process_step,date_time,step_num,previous_step_num,step_direction,time_on_each_step,next_step_num,jump_size
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,4,NaN,first_visit,NaN,4.0,0.0
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,4,4.0,repeat,52.0,NaN,NaN
2,9056452,306992881_89423906595,1000165_4190026492_760066,start,2017-06-04 01:07:29,0,NaN,first_visit,NaN,1.0,1.0
3,9056452,306992881_89423906595,1000165_4190026492_760066,step_1,2017-06-04 01:07:32,1,0.0,forward,3.0,2.0,1.0
4,9056452,306992881_89423906595,1000165_4190026492_760066,step_2,2017-06-04 01:07:56,2,1.0,forward,24.0,3.0,1.0
5,9056452,306992881_89423906595,1000165_4190026492_760066,step_3,2017-06-04 01:09:13,3,2.0,forward,77.0,4.0,1.0
6,9056452,306992881_89423906595,1000165_4190026492_760066,confirm,2017-06-04 01:09:50,4,3.0,forward,37.0,NaN,NaN
7,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,0,NaN,first_visit,NaN,1.0,1.0
8,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,1,0.0,forward,16.0,2.0,1.0
9,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,2,1.0,forward,9.0,1.0,-1.0


## Aggregate visit-level KPIs to client level

In [7]:
summary = df_visits.groupby("client_id").agg(
    session_per_client =("visit_id", "count"),
    max_step_client = ("max_step_reached", "max"),
    completion_rate = ("smooth_visit", "any"),
    avg_steps_client = ("n_steps_session", "mean"),
    error_rate=("had_error", "any")).reset_index()

summary


,client_id,session_per_client,max_step_client,completion_rate,avg_steps_client,error_rate
0,169,1,4,True,5.000000,False
1,336,1,0,False,2.000000,True
2,546,1,4,True,5.000000,False
3,555,1,4,True,5.000000,False
4,647,1,4,True,5.000000,False
...,...,...,...,...,...,...
120152,9999729,3,4,True,3.666667,True
120153,9999768,1,4,False,12.000000,True
120154,9999832,1,1,False,2.000000,False
120155,9999839,1,4,False,6.000000,True


### Average time to completion

Computed separately because it needs to filter to *completed* visits only. Averaging over all visits (including abandoned ones) would be meaningless. Clients who never completed will end up with `NaN` here, which is correct.

In [8]:
time_completion = df_visits[df_visits["smooth_visit"]].groupby("client_id").agg(
    avg_time_to_completion = ("visit_duration_sec", "mean")).reset_index()

time_completion

,client_id,avg_time_to_completion
0,169,213.0
1,546,133.0
2,555,158.0
3,647,377.0
4,1039,207.0
...,...,...
45909,9998851,165.0
45910,9999009,171.0
45911,9999400,119.0
45912,9999451,168.0


## Merge KPIs into the client roster

`how='left'` keeps every client in the roster, even if they have no visits or no completions.

In [9]:
summary_table_client = df_clients_roster.merge(summary, on='client_id', how='left').merge(time_completion, on='client_id', how='left')

In [10]:
summary_table_client

,client_id,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,session_per_client,max_step_client,completion_rate,avg_steps_client,error_rate,avg_time_to_completion
0,836976,73.0,60.5,U,2.0,45105.30,6.0,9.0,Test,2,4,False,5.500000,True,NaN
1,2304905,94.0,58.0,U,2.0,110860.30,6.0,9.0,Control,1,4,False,6.000000,True,NaN
2,1439522,64.0,32.0,U,2.0,52467.79,6.0,9.0,Test,2,3,False,2.500000,False,NaN
3,1562045,198.0,49.0,M,2.0,67454.65,3.0,6.0,Test,1,0,False,1.000000,False,NaN
4,5126305,145.0,33.0,F,2.0,103671.75,0.0,3.0,Control,1,0,False,1.000000,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50482,1780858,262.0,68.5,M,3.0,372100.59,6.0,9.0,Test,2,4,False,6.000000,True,NaN
50483,6967120,260.0,68.5,M,3.0,4279873.38,6.0,9.0,Control,1,4,True,5.000000,False,199.0
50484,5826160,249.0,56.5,F,2.0,44837.16,2.0,5.0,Test,3,4,False,3.333333,True,NaN
50485,8739285,229.0,69.5,F,2.0,44994.24,1.0,4.0,Test,1,4,True,5.000000,False,692.0


## Save

In [12]:
summary_table_client.to_csv("client_kpi_table.csv", index=False)